## DERIVATIVE PRICING
MODULE 6 | LESSON 4


---


# **LOCAL VOLATILITY MODELS: CEV IN PRACTICE**



|  |  |
|:---|:---|
|**Reading Time** |  55 minutes |
|**Prior Knowledge** | Local-volatility, CEV, Calibration, Implied volatility |
|**Keywords** | CEV model, Calibration|


---


*In the previous lesson, we covered the theoretical framework for one of the most famous parametric local volatility models: the CEV (constant elasticity of variance). In this notebook, we are going to see not only how to implement the CEV model in Python but also how to properly calibrate the model to the implied volatility observed from option market prices.*

*As usual, let's start by importing and installing the necessary libraries.* 

**Note:** Please make sure that you gather data on options chains from Yahoo finance when the U.S. market is open. Otherwise, you may encounter errors in the code when retrieving the data.


In [1]:
from datetime import date

import yahoo_fin.stock_info as si
from yahoo_fin import options

## **1. Download Option Prices**

\
Next, let's use the same features for extracting options chain data that we have worked on in Lessons 1 and 2 of this module. Please remember to set a maturity date for at least three months after the current date. If not, you may encounter some problems down the road related to the numerical optimization of the model/calibration (this has to do with issues related to the Jacobian).

In [2]:
# Set maturity date in at least 3 months to avoid numerical optimization problems:
Mat = date(2022, 8, 26)
T = Mat - date.today()
ticker = "IBM"
chain = options.get_options_chain(ticker, Mat)

callData = chain["calls"]
callData.head()

,Contract Name,Last Trade Date (EDT),Strike,Last Price,Bid,Ask,Change,% Change,Volume,Open Interest,Implied Volatility
0,There are no calls.,There are no calls.,There are no calls.,There are no calls.,There are no calls.,There are no calls.,There are no calls.,There are no calls.,There are no calls.,There are no calls.,There are no calls.


With this info, let's plot the market prices for the call options as a function of the different strikes:

In [3]:
import matplotlib.pyplot as plt

df_call = callData

df_call["Implied Volatility"] = df_call["Implied Volatility"].str[:-1]
df_call["Implied Volatility"] = df_call["Implied Volatility"].astype(float)

df_call.plot(kind="scatter", x="Strike", y="Last Price", color="red")
plt.show()

ValueError: could not convert string to float: 'There are no calls'

Obviously, as the option is more ITM, the premium of the call option increases. The question at this point is, can we replicate these prices with the CEV model?


## **2. Implementing CEV Model with Known Parameters**

Next, let's code our CEV model. As we have seen in the slides from the previous lesson, there are several equivalent specifications for the CEV model. We are going to follow the one in Hsu et al.,2008. (You can check the paper in the [additional readings for the lesson.](https://www.sciencedirect.com/science/article/pii/S0378475407002601?casa_token=x6aiAwSGHU0AAAAA:SWPmucDQ5mjSdSj5lzLRFgoPHGjq0L54Fs84zAmk9WfVjMGREQTJrDj-HGUX6e6iD5nSJcA5*) **Note**: This is not a required reading.)


Hsu et al.'s paper derives the following functional form for the call option price based on the following diffusion for the underlying asset:

$dS = \mu(S,t) dt + \sigma(S, t)dZ$, with:

$\sigma(S, t) = \sigma S^{\beta/2}$, $0\leq \beta < 2$

$\mu(S, t) = rS$

\
Of course, an important assumption is going to be the choice of our parameters $σ$ and and $\beta$. We will refine these choices later, but so far, let's just assume some given parameters. Later on, we will calibrate these parameters to market prices. For now:

- $\sigma = 0.35$
- $\beta = 1.25$

Also, let's assume a value for the risk-free rate:

- $r=0.05$


In [4]:
import numpy as np
from scipy.stats import ncx2
from sklearn.metrics import mean_squared_error

# Variables
S0 = si.get_live_price(ticker)
r = 0.05
Td = T.days / 365

sigma = 0.35
beta = 1.25


def C(t, K, sigma, beta):
    zb = 2 + 2 / (2 - beta)
    kappa = 2 * r / (sigma**2 * (2 - beta) * (np.exp(r * (2 - beta) * t) - 1))
    x = kappa * S0 ** (2 - beta) * np.exp(r * (2 - beta) * t)
    y = kappa * K ** (2 - beta)
    return S0 * (1 - ncx2.cdf(2 * y, zb, 2 * x)) - K * np.exp(-r * t) * (
        ncx2.cdf(2 * x, zb - 2, 2 * y)
    )


test_strikes = df_call["Strike"]
modelprices = C(Td, test_strikes, sigma, beta)
realprices = df_call["Last Price"]
plt.plot(test_strikes, modelprices, "o", label="Model")
plt.plot(test_strikes, realprices, "o", label="Real")
plt.xlabel("Stike")
plt.ylabel("Option price")
plt.legend()
err = mean_squared_error(modelprices.values, realprices)
print("Mean Squared Error is ", err)

TypeError: unsupported operand type(s) for ** or pow(): 'str' and 'float'

As you can see from the previous graphs, it seems that our model is not doing a very good job in replicating the observed option prices. (You even observe some very negative option prices.) Is it because of the functional form of the model, or is it just that we did not choose our parameters wisely enough?

## **3. CEV Model Calibration**

Finally, what we are going to do is calibrate this model to the option prices observed in the market. In other words, we are going to minimize the error between our CEV model prices and those prices observed in the market. We are going to optimize by changing only the parameters sigma ($\sigma$) and beta ($\beta$) in our CEV model. Hence, our minimization process will output the parameters sigma and beta that make the error with current market prices lower. This whole process is known as **calibration** of the model. 

- Why do we only focus on these parameters? Remember, **risk-neutral valuation**.

We will import the minimize module from scipy in order to proceed with the optimization. For now, we will perform a relatively simple minimization with the default procedures in scipy. In the future, you will see that some more complex calibrations may require devoting more time to the most suitable optimization method.<span style='color: transparent; font-size:1%'>All rights reserved WQU WorldQuant University QQQQ</span>

In [5]:
from scipy.optimize import minimize

We define our error function as the **mean squared error (MSE)** between model and market prices. This error function is what we will actually minimize.

In [6]:
def error(params):
    sigma = params[0]
    beta = params[1]
    modelprices = C(Td, test_strikes, sigma, beta)
    realprices = df_call["Last Price"]

    return mean_squared_error(modelprices, realprices)


bnds = ((0, None), (0, None))
res = minimize(
    error, (0.65, 1.8), bounds=bnds
)  # We will establish some initial guesses here (careful with altering this too much!)
print("Optimization process results:")
print(res)

TypeError: unsupported operand type(s) for ** or pow(): 'str' and 'float'

You can observe the results from the optimization process. The mean squared error (MSE) is clearly reduced from the initial scenario.

Now, let's see how the results from the model under the optimized parameters ($\sigma$ and $\beta$) look in a graph versus the real market prices:

In [7]:
modelprices = C(Td, test_strikes, res.x[0], res.x[1])
realprices = df_call["Last Price"]
plt.plot(test_strikes, modelprices, "o", label="Model")
plt.plot(test_strikes, realprices, "o", label="Real")
plt.xlabel("Stike")
plt.ylabel("Option price")
plt.legend()

NameError: name 'res' is not defined

## **4. Conclusion**

Well done! Now you know how to implement a CEV model and calibrate it to option market prices. This is one of the most important jobs for a quant. In the next module, we will continue to work on these ideas, extending the framework to consider the famous stochastic volatility model of Heston.

**References**


*   Hsu, Y. L. et al. "Constant Elasticity of Variance (CEV) Option Pricing Model: Integration and Detailed Variation." Mathematics and Computers in Simulation, vol. 79, no. 1, 2008, pp. 60–71.



---
Copyright 2023 WorldQuant University. This
content is licensed solely for personal use. Redistribution or
publication of this material is strictly prohibited.
